# Diagnostic audit: what did the Mario agents actually learn?

No training. Every test runs on the saved checkpoints of `Mario_RL_Attention_Research.ipynb` (v1, `checkpoints/`) and `Mario_RL_Generalization.ipynb` (v2, `checkpoints_generalization/`), and writes to `results_audit/`.

| Section | Test | Question |
|---|---|---|
| 4 | No-op starts, sticky actions, frozen HUD, 6-px shift | Is the policy closed-loop, or a memorised, timed button sequence? |
| 4 | Open-loop replay of the agent's own best episode | Does the agent beat a blind replay of itself when timing is perturbed? |
| 5 | Linear level-identity probe (HUD hidden) | Do the encoders memorise *which level* they are on? |
| 6 | Attention entropy and HUD attention on identical frames | Does narrow training make attention diffuse? Does attention go to the clock? |
| 7 | Occluding the most-attended vs random regions | Does the policy depend on what it attends to? |
| 8 | Hiding the HUD, shift-ensembling, greedy actions, attention off — at test time | Can unseen-level play be improved without retraining? |
| 9 | Correlating training-level measurements with unseen-level score | Can memorisation be detected without test levels? |

Run headless with `NOTEBOOK=Mario_RL_Audit.ipynb ./run_experiments.sh`. Resumable: finished evaluations are saved in `results_audit/raw/` and skipped on a re-run.

## 1. Setup: versions, parameters & output folders

In [ ]:
import importlib.util, subprocess, sys

NEW_MARIO_STACK = sys.version_info >= (3, 13)
if any(importlib.util.find_spec(m) is None for m in ("stable_baselines3", "gym_super_mario_bros", "nes_py", "scipy")):
    pkgs = ["stable-baselines3", "pandas", "matplotlib", "pillow", "scipy"]
    pkgs += (["gym-super-mario-bros>=9.1"] if NEW_MARIO_STACK else
             ["gym-super-mario-bros<8", "nes-py<9", "gym==0.25.2", "numpy==1.26.4"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

In [ ]:
import json, multiprocessing as mp, os, platform, re, time, warnings
from importlib.metadata import version as pkg_version, PackageNotFoundError
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import torch
import torch.nn.functional as F
import gymnasium
import stable_baselines3 as sb3
from stable_baselines3 import PPO
from scipy import stats
from IPython.display import display

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message=".*out of date.*")
warnings.filterwarnings("ignore", message=".*step API.*")

def _v(pkg):
    try:
        return pkg_version(pkg)
    except PackageNotFoundError:
        return None

VERSIONS = {"python": platform.python_version(), "numpy": np.__version__, "torch": torch.__version__,
            "gymnasium": gymnasium.__version__, "stable_baselines3": sb3.__version__,
            "gym_super_mario_bros": _v("gym-super-mario-bros"), "nes_py": _v("nes-py"), "gym": _v("gym")}
LEGACY_API = int(VERSIONS["nes_py"].split(".")[0]) < 9
print(" | ".join(f"{k} {v}" for k, v in VERSIONS.items() if v))
print("CPU cores:", os.cpu_count())
if LEGACY_API and not np.__version__.startswith("1."):
    raise RuntimeError("nes-py < 9 needs NumPy 1.x: pip install numpy==1.26.4, then restart the kernel")

In [ ]:
# Parameters: papermill overrides these, e.g. -p V1_EPISODES 5 -p N_WORKERS 6
V1_CKPT_DIR = "checkpoints"                 # v1: checkpoints/<level>/<variant>_s<seed>/
V2_CKPT_DIR = "checkpoints_generalization"  # v2: checkpoints_generalization/<variant>_s<seed>/
V1_LEVELS = ["1-1", "1-2", "1-3"]           # every v1 model plays all of these (its own level + the unseen ones)
V2_TRAIN_LEVELS = None                      # None: read from results_generalization/config.json
V2_TEST_LEVELS = None
CONDITIONS = ["clean", "noop", "sticky", "hud_freeze", "shift"]
REPLAY = True                               # open-loop replay of the agent's best clean episode, with no-op starts
INTERVENTIONS = ["hud_mask", "tta", "greedy", "attn_off"]   # test-time changes tried on unseen levels (section 8)
TTA_SHIFT = 4                               # test-time augmentation: average the policy over 5 views shifted by this many pixels
DORMANT_TAU = 0.1                           # a neuron is dormant if its normalised mean activation ≤ τ (Sokar et al., 2023)
V1_EPISODES = 10                            # per model, level and condition
V2_EPISODES = 4
RANDOM_EPISODES = 10                        # random-policy reference per level
MAX_STEPS = 3000                            # agent decisions per episode
STALL_STEPS = 250                           # end the episode after this many decisions without new progress (≈ 17 s)
NOOP_MAX = 30                               # no-op starts: 0..NOOP_MAX idle decisions (up to 2 s) before the agent acts
STICKY_P = 0.25                             # sticky actions: probability the previous action repeats (Machado et al., 2018)
SHIFT_PX = 6                                # whole-screen horizontal shift in pixels
PROBE_EPISODES = 6                          # probe frames: episodes per level (also the cross-validation groups)
PROBE_EVERY = 3                             # keep every 3rd decision
PROBE_MAX_PER_EPISODE = 30
OCCLUDE_K = 8                               # feature-map cells occluded (of 11 × 12 = 132)
OCCLUDE_FRAMES = 30                         # frames per level for the occlusion test
N_WORKERS = None                            # None: CPU cores − 1 (at most 12)
REEVALUATE = False                          # True: ignore saved evaluations in results_audit/raw/

In [ ]:
TITLE = "Diagnostic audit of pixel-based PPO agents on Super Mario Bros."
ROOT = Path.cwd()
V1_CKPT, V2_CKPT = ROOT / V1_CKPT_DIR, ROOT / V2_CKPT_DIR
RESULTS = ROOT / "results_audit"
DIRS = {k: RESULTS / k for k in ["figures", "tables", "raw"]}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)
(DIRS["raw"] / "robustness").mkdir(exist_ok=True)
(DIRS["raw"] / "interventions").mkdir(exist_ok=True)

v2_cfg_file = ROOT / "results_generalization" / "config.json"
v2_cfg = json.loads(v2_cfg_file.read_text())["config"] if v2_cfg_file.exists() else {}
V2_TRAIN_LEVELS = V2_TRAIN_LEVELS or v2_cfg.get("train_levels") or [
    "1-1", "1-2", "2-3", "3-1", "3-3", "4-1", "4-3", "5-1", "5-2", "6-1", "6-2", "6-3", "7-1", "7-3", "8-2", "8-3"]
V2_TEST_LEVELS = V2_TEST_LEVELS or v2_cfg.get("test_levels") or ["1-3", "2-1", "3-2", "4-2", "5-3", "8-1"]
N_WORKERS = N_WORKERS or max(1, min(12, (os.cpu_count() or 2) - 1))
EP_SEED = 10_000                                  # episode k of every model and condition uses seed EP_SEED + k

CONFIG = {k: globals()[k] for k in ["V1_LEVELS", "V2_TRAIN_LEVELS", "V2_TEST_LEVELS", "CONDITIONS", "REPLAY", "INTERVENTIONS",
                                    "TTA_SHIFT", "DORMANT_TAU", "V1_EPISODES",
                                    "V2_EPISODES", "RANDOM_EPISODES", "MAX_STEPS", "STALL_STEPS", "NOOP_MAX", "STICKY_P",
                                    "SHIFT_PX", "PROBE_EPISODES", "PROBE_EVERY", "PROBE_MAX_PER_EPISODE", "OCCLUDE_K",
                                    "OCCLUDE_FRAMES", "N_WORKERS", "EP_SEED"]}
(RESULTS / "config.json").write_text(json.dumps({"title": TITLE, "config": CONFIG, "environment": VERSIONS}, indent=2))

COND_LABELS = {"clean": "Clean", "noop": "No-op start", "sticky": "Sticky actions", "hud_freeze": "Frozen HUD",
               "shift": f"{SHIFT_PX}-px shift", "replay_noop": "Open-loop replay + no-op start",
               "tape_noop": "Blind tape + no-op start", "tape_sticky": "Blind tape + sticky actions",
               "hud_mask": "HUD hidden", "tta": "Shift-ensemble (TTA)", "greedy": "Greedy actions", "attn_off": "Attention switched off"}
VARIANT_LABELS = {"baseline": "NatureCNN", "channel": "+ SE", "spatial": "+ spatial attn", "pointwise": "+ pointwise",
                  "cnn": "NatureCNN", "cnn_aug": "NatureCNN + aug", "attn": "Dense attn", "attn_aug": "Dense attn + aug",
                  "sparse": "Sparse attn", "sparse_aug": "Sparse attn + aug", "impala_aug": "IMPALA + aug"}
GROUPS = ["v1 · trained level", "v1 · unseen level", "v2 · training levels", "v2 · held-out levels"]
GROUP_STYLE = {"v1 · trained level": ("#2a78d6", None), "v1 · unseen level": ("#a9c9f0", "//"),
               "v2 · training levels": ("#1baf7a", None), "v2 · held-out levels": ("#a6e0c8", "//")}

plt.rcParams.update({"figure.dpi": 100, "savefig.dpi": 300, "axes.spines.top": False, "axes.spines.right": False})

def save_fig(fig, name, formats=("png", "pdf")):
    for ext in formats:
        fig.savefig(DIRS["figures"] / f"{name}.{ext}", bbox_inches="tight")
    print(f"saved → {RESULTS.name}/figures/{name}.{{{','.join(formats)}}}")

def _fmt(v, floatfmt):
    if v is None or (isinstance(v, (float, np.floating)) and np.isnan(v)):
        return "–"
    if isinstance(v, (bool, np.bool_)):
        return "yes" if v else "no"
    if isinstance(v, (float, np.floating)):
        return format(v, floatfmt)
    if isinstance(v, (int, np.integer)):
        return f"{v:,}"
    return str(v)

def df_to_markdown(df, floatfmt=",.2f"):
    head = "| " + " | ".join(map(str, df.columns)) + " |"
    rule = "|" + "|".join([":---"] + ["---:"] * (len(df.columns) - 1)) + "|"
    return "\n".join([head, rule, *["| " + " | ".join(_fmt(v, floatfmt) for v in r) + " |" for r in df.itertuples(index=False)]])

def df_to_latex(df, floatfmt=",.2f", caption="", label=""):
    def esc(s):
        s = str(s)
        for a, b in [("\\", r"\textbackslash{}"), ("&", r"\&"), ("%", r"\%"), ("_", r"\_"), ("#", r"\#"),
                     ("±", r"$\pm$"), ("≥", r"$\geq$"), ("×", r"$\times$"), ("−", r"$-$"), ("–", "--"),
                     ("→", r"$\rightarrow$"), ("·", r"$\cdot$"), ("‖", r"$\|$"), ("γ", r"$\gamma$")]:
            s = s.replace(a, b)
        return s
    body = "\n".join(" & ".join(esc(_fmt(v, floatfmt)) for v in r) + r" \\" for r in df.itertuples(index=False))
    return ("\\begin{table}[t]\n\\centering\n" f"\\caption{{{esc(caption)}}}\n\\label{{tab:{label}}}\n"
            f"\\begin{{tabular}}{{l{'r' * (len(df.columns) - 1)}}}\n\\toprule\n"
            + " & ".join(esc(c) for c in df.columns) + " \\\\\n\\midrule\n" + body + "\n\\bottomrule\n\\end{tabular}\n\\end{table}\n")

def save_table(df, name, caption=None, floatfmt=",.2f"):
    caption = caption or name
    df.to_csv(DIRS["tables"] / f"{name}.csv", index=False)
    (DIRS["tables"] / f"{name}.md").write_text(f"### {caption}\n\n{df_to_markdown(df, floatfmt)}\n")
    (DIRS["tables"] / f"{name}.tex").write_text(df_to_latex(df, floatfmt, caption, name))
    print(f"saved → {RESULTS.name}/tables/{name}.{{csv,md,tex}}")
    display(df)

def boot_ci(x, n=10_000, seed=0):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    if len(x) < 2:
        return (np.nan, np.nan)
    m = np.random.default_rng(seed).choice(x, (n, len(x))).mean(1)
    return tuple(np.percentile(m, [2.5, 97.5]))

def mean_ci(x, fmt="{:.2f}"):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    if len(x) == 0:
        return "–"
    lo, hi = boot_ci(x)
    return fmt.format(x.mean()) if np.isnan(lo) else f"{fmt.format(x.mean())} ({fmt.format(lo)}–{fmt.format(hi)})"

def wilcoxon_p(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    ok = ~(np.isnan(a) | np.isnan(b)); a, b = a[ok], b[ok]
    if len(a) < 2 or np.allclose(a, b):
        return np.nan
    try:
        return float(stats.wilcoxon(a, b).pvalue)
    except ValueError:
        return np.nan

def holm(p):
    p = np.asarray(p, float); out = np.full_like(p, np.nan); ok = ~np.isnan(p)
    order = np.argsort(p[ok]); m = ok.sum()
    out[np.flatnonzero(ok)[order]] = np.maximum.accumulate((m - np.arange(m)) * p[ok][order]).clip(max=1)
    return out

print(f"results → {RESULTS} | workers: {N_WORKERS}")

## 2. Environment, perturbations and evaluation loop

In [ ]:
import gym_super_mario_bros
from gym_super_mario_bros.actions import SIMPLE_MOVEMENT
from nes_py.wrappers import JoypadSpace

N_ACTIONS, FRAME_SKIP, HUD_ROWS = len(SIMPLE_MOVEMENT), 4, 16   # rows 0-15 of the 120×128 frame: score, coins, WORLD, TIME

def make_mario(level):
    return JoypadSpace(gym_super_mario_bros.make(f"SuperMarioBros-{level}-v0"), SIMPLE_MOVEMENT)

def mario_reset(env):
    out = env.reset()
    return out[0] if isinstance(out, tuple) else out

def env_step(env, action):
    """One agent decision = FRAME_SKIP emulator frames, as in training. Returns (frame, info, done)."""
    for _ in range(FRAME_SKIP):
        out = env.step(action)
        frame, info, done = (out[0], out[3], out[2]) if len(out) == 4 else (out[0], out[4], out[2] or out[3])
        if done:
            break
    return frame, info, bool(done)

def prep(frame):
    small = frame[::2, ::2].astype(np.float32)                          # 240×256 → 120×128 grayscale, as in training
    return (small[..., 0] * 0.299 + small[..., 1] * 0.587 + small[..., 2] * 0.114).astype(np.uint8)

def shift_frame(g, px):
    """Move the picture px pixels to the right; the left edge is replicated."""
    return g if px == 0 else np.concatenate([np.repeat(g[:, :1], px, axis=1), g[:, :-px]], axis=1)

def shift2d(x, dy, dx):
    """Move an (H, W, C) image by (dy, dx) pixels; edges replicated."""
    pad = max(abs(dy), abs(dx))
    if pad == 0:
        return x
    xp = np.pad(x, ((pad, pad), (pad, pad), (0, 0)), mode="edge")
    return xp[pad - dy:pad - dy + x.shape[0], pad - dx:pad - dx + x.shape[1]]

def mask_hud(stacks):
    """Paint the HUD rows with the background colour (row 0) in every stacked frame."""
    out = stacks.copy()
    out[..., :HUD_ROWS, :, :] = out[..., :1, :, :]
    return out


class Agent:
    """A saved policy, run on CPU with stochastic actions (the evaluation protocol of v1 and v2)."""
    def __init__(self, path):
        self.model = PPO.load(path, device="cpu")
        self.policy = self.model.policy
        self.policy.set_training_mode(False)

    @torch.no_grad()
    def probs(self, stacks):                                            # (N, 120, 128, 4) uint8 → (N, 7)
        t, _ = self.policy.obs_to_tensor(stacks)
        return self.policy.get_distribution(t).distribution.probs.cpu().numpy().astype(np.float64)

    def act(self, stack, rng):
        p = self.probs(stack[None])[0]
        return int(rng.choice(N_ACTIONS, p=p / p.sum()))

    def act_greedy(self, stack, rng):
        return int(self.probs(stack[None])[0].argmax())

    def act_tta(self, stack, rng):
        """Test-time augmentation: average the action probabilities over the frame and 4 shifted copies."""
        k = TTA_SHIFT
        views = np.stack([shift2d(stack, dy, dx) for dy, dx in [(0, 0), (k, 0), (-k, 0), (0, k), (0, -k)]])
        p = self.probs(views).mean(0)
        return int(rng.choice(N_ACTIONS, p=p / p.sum()))

    def attention_scale(self, scale):
        """Multiply the attention block's residual gate γ (0 = block switched off, 1 = as trained)."""
        blk = getattr(self.policy.features_extractor, "block", None)
        if blk is not None and hasattr(blk, "gamma"):
            if not hasattr(self, "_gamma0"):
                self._gamma0 = blk.gamma.data.clone()
            blk.gamma.data = self._gamma0 * scale


def run_episode(env, condition, ep_seed, act=None, replay=None):
    """One life on one level under a condition. act(stack, rng) → action; or replay = a fixed action list (open loop).
    Episode k of every model and condition draws the same no-op count (common random numbers)."""
    rng = np.random.default_rng(ep_seed)
    n_noop = int(rng.integers(0, NOOP_MAX + 1))
    n_noop = n_noop if condition in ("noop", "replay_noop", "tape_noop") else 0
    frame = prep(mario_reset(env)); hud0 = frame[:HUD_ROWS].copy()
    stack = np.zeros((120, 128, 4), np.uint8)

    def push(f):
        if condition == "hud_freeze":
            f = f.copy(); f[:HUD_ROWS] = hud0
        elif condition == "shift":
            f = shift_frame(f, SHIFT_PX)
        elif condition == "hud_mask":
            f = f.copy(); f[:HUD_ROWS] = f[0]
        stack[..., :-1] = stack[..., 1:]; stack[..., -1] = f

    push(frame)
    max_x, last_gain, prev, actions, info, ended = 0, 0, 0, [], {}, "timeout"
    for t in range(MAX_STEPS):
        if t < n_noop:
            a = 0
        elif replay is not None:
            if t - n_noop >= len(replay):
                ended = "replay_end"; break
            a = replay[t - n_noop]
            if condition == "tape_sticky" and t > 0 and rng.random() < STICKY_P:
                a = prev
        else:
            a = act(stack, rng)
            if condition == "sticky" and t > 0 and rng.random() < STICKY_P:
                a = prev
        prev = a; actions.append(a)
        frame, info, done = env_step(env, a)
        push(prep(frame))
        x = int(info.get("x_pos", 0))
        if x > max_x:
            max_x, last_gain = x, t
        if done:
            ended = "flag" if info.get("flag_get", False) else "death"; break
        if t - last_gain >= STALL_STEPS:
            ended = "stall"; break
    row = {"condition": condition, "max_x_pos": max_x, "steps": t + 1, "n_noop": n_noop,
           "flag_get": int(bool(info.get("flag_get", False))), "ended": ended}
    return row, actions


def eval_job(job):
    """All conditions for one (model, level) pair; saved to results_audit/raw/<subdir>/<id>.csv."""
    torch.set_num_threads(1)
    out = DIRS["raw"] / job.get("subdir", "robustness") / f"{job['id']}.csv"
    if out.exists() and not REEVALUATE:
        return pd.read_csv(out, dtype={"level": str})
    env = make_mario(job["level"])
    agent = None if job["path"] is None else Agent(job["path"])
    rows, best, tapes = [], (-1, None), {}
    for cond in job["conditions"]:
        if agent is None:
            act = lambda stack, rng: int(rng.integers(N_ACTIONS))
        else:
            act = {"tta": agent.act_tta, "greedy": agent.act_greedy}.get(cond, agent.act)
            agent.attention_scale(0.0 if cond == "attn_off" else 1.0)
        for k in range(job["episodes"]):
            if cond == "replay_noop":
                row, _ = run_episode(env, cond, EP_SEED + k, replay=best[1])
            elif cond.startswith("tape_"):                  # episode k's own clean button tape, replayed blind
                row, _ = run_episode(env, cond, EP_SEED + k, replay=tapes[k])
            else:
                row, actions = run_episode(env, cond, EP_SEED + k, act=act)
                if cond == "clean":
                    tapes[k] = actions
                    if row["max_x_pos"] > best[0]:
                        best = (row["max_x_pos"], actions)
            rows.append({**{c: job[c] for c in ["model", "family", "variant", "seed", "level", "split"]},
                         "episode": k, **row, "replay_of_x": best[0] if cond == "replay_noop" else np.nan})
    env.close()
    df = pd.DataFrame(rows)
    df.to_csv(out.with_suffix(".tmp"), index=False); out.with_suffix(".tmp").rename(out)
    return df


def run_pool(fn, jobs, label):
    """Runs jobs in N_WORKERS forked processes with a progress line; finished jobs are cached by fn itself."""
    if not jobs:
        return []
    t0, results = time.time(), []
    ctx = mp.get_context("fork")
    pool = ctx.Pool(min(N_WORKERS, len(jobs)), maxtasksperchild=8)
    try:
        for i, res in enumerate(pool.imap_unordered(fn, jobs), 1):
            results.append(res)
            el = time.time() - t0
            if i == len(jobs) or i % max(1, len(jobs) // 20) == 0:
                print(f"[{label}] {i}/{len(jobs)} done | {el / 60:.1f} min elapsed | ~{el / i * (len(jobs) - i) / 60:.1f} min left",
                      flush=True)
        pool.close()
    except BaseException:
        pool.terminate()
        print(f"[{label}] interrupted: finished jobs are saved; re-run to continue")
        raise
    finally:
        pool.join()
    return results

# perturbation examples
e = make_mario("1-1"); f0 = prep(mario_reset(e))
for _ in range(30):
    f1, _, done = env_step(e, 3)                   # run right for 2 s
    if done:
        break
e.close(); f1 = prep(f1)
frozen = f1.copy(); frozen[:HUD_ROWS] = f0[:HUD_ROWS]
fig, axes = plt.subplots(1, 4, figsize=(13, 3))
for ax, (img, t) in zip(axes, [(f1, "clean"), (frozen, "frozen HUD: clock stopped at 0 s"),
                               (shift_frame(f1, SHIFT_PX), f"{SHIFT_PX}-px shift"), (mask_hud(f1[None, ..., None])[0, ..., 0], "HUD hidden (probe only)")]):
    ax.imshow(img, cmap="gray", vmin=0, vmax=255); ax.set_title(t, fontsize=9); ax.axis("off")
fig.tight_layout(); save_fig(fig, "perturbation_examples"); plt.show()

## 3. Saved models

In [ ]:
def latest_ckpt(d):
    finals = sorted(d.glob("*_final.zip"))
    if finals:
        return finals[0], True
    steps = sorted((int(m.group(1)), p) for p in d.glob("*.zip") if (m := re.search(r"_(\d+)_steps\.zip$", p.name)))
    return (steps[-1][1], False) if steps else (None, False)

rows = []
if V1_CKPT.exists():
    for lvl_dir in sorted(p for p in V1_CKPT.iterdir() if p.is_dir()):
        for run_dir in sorted(p for p in lvl_dir.iterdir() if p.is_dir()):
            m = re.match(r"(.+)_s(\d+)$", run_dir.name); path, final = latest_ckpt(run_dir)
            if m and path:
                rows.append({"model": f"v1_{lvl_dir.name}_{run_dir.name}", "family": "v1", "variant": m.group(1),
                             "seed": int(m.group(2)), "trained_levels": [lvl_dir.name], "path": str(path), "final": final})
if V2_CKPT.exists():
    for run_dir in sorted(p for p in V2_CKPT.iterdir() if p.is_dir()):
        m = re.match(r"(.+)_s(\d+)$", run_dir.name); path, final = latest_ckpt(run_dir)
        if m and path:
            rows.append({"model": f"v2_{run_dir.name}", "family": "v2", "variant": m.group(1), "seed": int(m.group(2)),
                         "trained_levels": list(V2_TRAIN_LEVELS), "path": str(path), "final": final})
MODELS = pd.DataFrame(rows)
assert len(MODELS), f"no checkpoints found in {V1_CKPT} or {V2_CKPT}"

info = []
for r in MODELS.itertuples():
    pol = PPO.load(r.path, device="cpu").policy
    blk = getattr(pol.features_extractor, "block", None)
    g = getattr(blk, "gamma", None)
    info.append({"block": type(blk).__name__ if blk is not None else "none", "has_attention": hasattr(blk, "store_attn"),
                 "gamma": None if g is None else float(g.detach().abs().max())})
MODELS = pd.concat([MODELS, pd.DataFrame(info)], axis=1)
ATTN_MODELS = MODELS[MODELS.has_attention].model.tolist()
ALL_LEVELS = list(dict.fromkeys(V1_LEVELS + (V2_TRAIN_LEVELS + V2_TEST_LEVELS if (MODELS.family == "v2").any() else [])))

save_table(MODELS.assign(trained_on=MODELS.trained_levels.map(lambda l: l[0] if len(l) == 1 else f"{len(l)} levels"),
                         checkpoint=MODELS.path.map(lambda p: Path(p).name))
           [["model", "family", "variant", "seed", "trained_on", "block", "gamma", "final", "checkpoint"]],
           "models", "Checkpoints audited", ",.4f")
if not MODELS.final.all():
    print("note: some runs have no final checkpoint; their latest step checkpoint is used")

## 4. Robustness: closed-loop control or a memorised button sequence?

Every model plays its training level(s) and unseen levels under five conditions; the same episode seeds are used for every model and condition.

| Condition | What changes | A memorising agent… |
|---|---|---|
| `noop` | 0–30 idle decisions before the agent takes control (Mnih et al., 2015) | is now out of sync with enemies and its own timing |
| `sticky` | each decision repeats the previous one with p = 0.25 (Machado et al., 2018) | cannot correct the drift |
| `hud_freeze` | the score/coin/clock bar is frozen at its first frame | loses its clock if it times moves off the HUD |
| `shift` | the screen is shifted 6 px to the right | breaks if it keys on exact pixel positions |
| `replay_noop` | the agent's own best clean episode, replayed blind with the same no-op starts | (reference: what pure memorisation achieves) |

**Retention** = mean max x under the condition ÷ mean max x in the clean condition, for each (model, level); 1.0 = unaffected. **Score** = max x ÷ random policy on that level. A closed-loop policy should keep most of its progress under `noop` and `sticky` and clearly beat its own open-loop replay.

In [ ]:
jobs = []
for r in MODELS.itertuples():
    levels = V1_LEVELS if r.family == "v1" else V2_TRAIN_LEVELS + V2_TEST_LEVELS
    for lvl in levels:
        trained = lvl in r.trained_levels
        split = ("trained level" if trained else "unseen level") if r.family == "v1" else \
                ("training levels" if trained else "held-out levels")
        conds = CONDITIONS + (["replay_noop"] if REPLAY and trained and "clean" in CONDITIONS else [])
        jobs.append({"id": f"{r.model}__{lvl}", "model": r.model, "family": r.family, "variant": r.variant, "seed": r.seed,
                     "level": lvl, "split": split, "path": r.path, "conditions": conds,
                     "episodes": V1_EPISODES if r.family == "v1" else V2_EPISODES})
for lvl in ALL_LEVELS:
    jobs.append({"id": f"random__{lvl}", "model": "random", "family": "random", "variant": "random", "seed": 0, "level": lvl,
                 "split": "reference", "path": None, "conditions": ["clean"], "episodes": RANDOM_EPISODES})
jobs.sort(key=lambda j: (j["family"] == "random", -len(j["conditions"]) * j["episodes"]))   # long jobs first
cached = sum((DIRS["raw"] / "robustness" / f"{j['id']}.csv").exists() for j in jobs) if not REEVALUATE else 0
print(f"{len(jobs)} (model, level) jobs, {cached} already saved; "
      f"{sum(len(j['conditions']) * j['episodes'] for j in jobs):,} episodes in total")
EPISODES = pd.concat(run_pool(eval_job, jobs, "robustness"), ignore_index=True)
EPISODES.to_csv(DIRS["raw"] / "robustness_episodes.csv", index=False)
RANDOM_MEAN = EPISODES[EPISODES.model == "random"].groupby("level").max_x_pos.mean().to_dict()
EPISODES["score"] = EPISODES.max_x_pos / EPISODES.level.map(RANDOM_MEAN)
EPISODES["group"] = EPISODES.family + " · " + EPISODES.split
print(f"saved → {RESULTS.name}/raw/robustness_episodes.csv ({len(EPISODES):,} episodes)")

In [ ]:
ep = EPISODES[EPISODES.model != "random"]
UNITS = ep.pivot_table(index=["model", "family", "variant", "seed", "level", "group"], columns="condition",
                       values="max_x_pos", aggfunc="mean").reset_index()
UNITS["clean_score"] = UNITS["clean"] / UNITS.level.map(RANDOM_MEAN)
PERTURB = [c for c in CONDITIONS if c != "clean"]
for c in PERTURB:
    UNITS[f"ret_{c}"] = UNITS[c] / UNITS["clean"]
UNITS.to_csv(DIRS["raw"] / "robustness_units.csv", index=False)
groups = [g for g in GROUPS if g in set(UNITS.group)]

rows = []
for g in groups:
    u = UNITS[UNITS.group == g]
    rows.append({"Group": g, "n (model × level)": len(u), "Clean score (× random)": mean_ci(u.clean_score),
                 **{COND_LABELS[c]: mean_ci(u[f"ret_{c}"]) for c in PERTURB}})
save_table(pd.DataFrame(rows), "robustness_summary",
           "Clean score and retention under each perturbation (mean, 95% bootstrap CI over model × level; 1.0 = unaffected)")

rows = []
for g in groups:
    u = UNITS[UNITS.group == g]
    for c in PERTURB:
        rows.append({"Group": g, "Condition": COND_LABELS[c], "n": len(u), "Mean retention": u[f"ret_{c}"].mean(),
                     "Units below 0.9": f"{(u[f'ret_{c}'] < 0.9).mean():.0%}", "p (Wilcoxon vs clean)": wilcoxon_p(u[c], u["clean"])})
st = pd.DataFrame(rows); st["p (Holm)"] = holm(st["p (Wilcoxon vs clean)"])
save_table(st, "robustness_statistics", "Perturbation vs clean max x, paired over model × level (Holm over all rows)", ",.3f")

rows = []
for (fam, var, grp), u in UNITS.groupby(["family", "variant", "group"], sort=False):
    rows.append({"Group": grp, "Variant": VARIANT_LABELS.get(var, var), "Models": u.model.nunique(),
                 "Clean score": u.clean_score.mean(), **{COND_LABELS[c]: u[f"ret_{c}"].mean() for c in PERTURB}})
pv = pd.DataFrame(rows); pv["_g"] = pv.Group.map(GROUPS.index); pv = pv.sort_values(["_g", "Variant"]).drop(columns="_g")
save_table(pv, "robustness_by_variant", "Clean score and mean retention per variant")

end = ep.groupby(["group", "condition"]).ended.value_counts(normalize=True).unstack(fill_value=0).reset_index()
end["_g"] = end.group.map(GROUPS.index); end["_c"] = end.condition.map(list(COND_LABELS).index)
end = end.sort_values(["_g", "_c"]).drop(columns=["_g", "_c"])
end["condition"] = end.condition.map(COND_LABELS)
save_table(end.rename(columns={"group": "Group", "condition": "Condition"}), "episode_endings",
           "How episodes ended (share): death, stall (no progress for STALL_STEPS decisions), flag, end of replay", ",.2f")

fig, ax = plt.subplots(figsize=(1.9 * len(PERTURB) + 1.5, 3.8))
w = 0.8 / len(groups)
for i, g in enumerate(groups):
    u = UNITS[UNITS.group == g]
    m = [u[f"ret_{c}"].mean() for c in PERTURB]
    ci = np.array([boot_ci(u[f"ret_{c}"]) for c in PERTURB])
    col, hatch = GROUP_STYLE[g]
    ax.bar(np.arange(len(PERTURB)) + (i - (len(groups) - 1) / 2) * w, m, w * 0.92, color=col, hatch=hatch,
           edgecolor="white", linewidth=0, label=g, yerr=np.abs(ci.T - m), capsize=2, error_kw={"elinewidth": 0.8})
ax.axhline(1, color="#555", lw=0.8, ls="--")
ax.set_xticks(range(len(PERTURB)), [COND_LABELS[c] for c in PERTURB])
ax.set_ylabel("retention (max x ÷ clean max x)")
ax.legend(frameon=False, fontsize=8, ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.12))
ax.set_title("Progress kept under each perturbation (95% CI)", fontsize=10)
fig.tight_layout(); save_fig(fig, "robustness_retention"); plt.show()

### Agent vs. its own open-loop replay

For each model on a level it was trained on, the best clean episode is replayed action by action without looking at the screen, after the same no-op start the agent received. NES Mario is deterministic, so without the no-op start this replay would reproduce the episode exactly. If the agent keeps no more progress than its blind replay, it is effectively executing a memorised sequence.

In [ ]:
rep = ep[ep.condition == "replay_noop"]
if len(rep):
    rr = rep.groupby(["model", "variant", "level", "group"]).agg(replay_x=("max_x_pos", "mean"), best_clean_x=("replay_of_x", "first")).reset_index()
    rr = rr.merge(UNITS[["model", "level", "clean", "noop"]], on=["model", "level"])
    rr["agent_retention"] = rr["noop"] / rr["clean"]
    rr["replay_retention"] = rr["replay_x"] / rr["best_clean_x"]
    rr.to_csv(DIRS["raw"] / "replay_units.csv", index=False)
    rows = []
    parts = [((g, "all"), rr[rr.group == g]) for g in rr.group.unique()] + list(rr.groupby(["group", "variant"]))
    for (g, v), u in parts:
        rows.append({"Group": g, "Variant": "all" if v == "all" else VARIANT_LABELS.get(v, v), "n": len(u),
                     "Agent retention (no-op)": u.agent_retention.mean(), "Open-loop replay retention (no-op)": u.replay_retention.mean(),
                     "Agent − replay": (u.agent_retention - u.replay_retention).mean(),
                     "Agent > replay": f"{(u.agent_retention > u.replay_retention).mean():.0%}",
                     "p (Wilcoxon)": wilcoxon_p(u.agent_retention, u.replay_retention)})
    save_table(pd.DataFrame(rows), "replay_comparison",
               "Agent vs. blind replay of its own best episode, both with no-op starts (retention = share of clean progress kept)", ",.2f")

    fig, ax = plt.subplots(figsize=(4.4, 4.2))
    for g in [g for g in GROUPS if g in set(rr.group)]:
        u = rr[rr.group == g]
        ax.scatter(u.replay_retention, u.agent_retention, s=26, color=GROUP_STYLE[g][0], label=g, alpha=0.85, edgecolor="white", lw=0.5)
    lim = [0, max(1.05, rr[["agent_retention", "replay_retention"]].max().max() * 1.05)]
    ax.plot(lim, lim, color="#555", lw=0.8, ls="--"); ax.set_xlim(lim); ax.set_ylim(lim)
    ax.text(lim[1] * 0.97, lim[1] * 0.06, "agent = blind replay", ha="right", fontsize=8, color="#555")
    ax.set_xlabel("open-loop replay retention"); ax.set_ylabel("agent retention")
    ax.set_title("Above the line: the agent reacts to the screen", fontsize=10); ax.legend(frameon=False, fontsize=8)
    fig.tight_layout(); save_fig(fig, "replay_vs_agent"); plt.show()
else:
    print("no replay episodes (REPLAY = False or no clean condition)")

### Closed-loop index: the agent vs. its own button tapes

The best-episode replay above is a weak test where no-op starts barely disturb a level: a blind tape then keeps most of its progress anyway. Here every clean episode's button sequence (its "tape") is replayed blind under the same perturbation the agent faced: a no-op start, or sticky actions. Both are compared with the same clean episodes, so they share one denominator:

**Closed-loop index (CLI)** = agent retention − tape retention. CLI ≈ 0: under that perturbation the agent does no better than a recording of itself, i.e. it is effectively open-loop. CLI > 0: it uses the screen to recover. The clean episodes are re-played first; NES Mario and the seeded policy are deterministic, so they must reproduce the section-4 clean results exactly (checked below).

In [ ]:
jobs = []
for r in MODELS.itertuples():
    for lvl in r.trained_levels if r.family == "v1" else V2_TRAIN_LEVELS:
        jobs.append({"id": f"{r.model}__{lvl}", "subdir": "tapes", "model": r.model, "family": r.family, "variant": r.variant,
                     "seed": r.seed, "level": lvl, "split": "trained level" if r.family == "v1" else "training levels",
                     "path": r.path, "conditions": ["clean", "tape_noop", "tape_sticky"],
                     "episodes": V1_EPISODES if r.family == "v1" else V2_EPISODES})
(DIRS["raw"] / "tapes").mkdir(exist_ok=True)
print(f"{len(jobs)} (model, training level) jobs, {sum(3 * j['episodes'] for j in jobs):,} episodes")
TAPES = pd.concat(run_pool(eval_job, jobs, "tapes"), ignore_index=True)
TAPES["group"] = TAPES.family + " · " + TAPES.split
TAPES.to_csv(DIRS["raw"] / "tape_episodes.csv", index=False)

chk = TAPES[TAPES.condition == "clean"].merge(EPISODES[EPISODES.condition == "clean"], on=["model", "level", "episode"], suffixes=("", "_s4"))
print(f"determinism check: {(chk.max_x_pos == chk.max_x_pos_s4).mean():.1%} of {len(chk)} clean episodes reproduced exactly")

tu = TAPES.pivot_table(index=["model", "variant", "level", "group"], columns="condition", values="max_x_pos", aggfunc="mean").reset_index()
tu = tu.merge(UNITS[["model", "level", "noop", "sticky"]], on=["model", "level"])
for c in ["noop", "sticky"]:
    tu[f"agent_{c}"] = tu[c] / tu["clean"]
    tu[f"tape_{c}_ret"] = tu[f"tape_{c}"] / tu["clean"]
    tu[f"cli_{c}"] = tu[f"agent_{c}"] - tu[f"tape_{c}_ret"]
tu.to_csv(DIRS["raw"] / "closed_loop_units.csv", index=False)
CLI = tu
rows = []
parts = [((g, "all"), tu[tu.group == g]) for g in [g for g in GROUPS if g in set(tu.group)]] + list(tu.groupby(["group", "variant"]))
for (g, v), u in parts:
    row = {"Group": g, "Variant": "all" if v == "all" else VARIANT_LABELS.get(v, v), "n": len(u)}
    for c, lab in [("noop", "No-op"), ("sticky", "Sticky")]:
        row.update({f"{lab}: agent": u[f"agent_{c}"].mean(), f"{lab}: tape": u[f"tape_{c}_ret"].mean(),
                    f"{lab}: CLI": mean_ci(u[f"cli_{c}"]), f"{lab}: p": wilcoxon_p(u[c], u[f"tape_{c}"])})
    rows.append(row)
t = pd.DataFrame(rows)
for lab in ["No-op", "Sticky"]:
    t[f"{lab}: p (Holm)"] = holm(t[f"{lab}: p"])
save_table(t, "closed_loop_index", "Closed-loop index: agent retention − blind-tape retention under the same perturbation (mean, 95% CI)", ",.2f")

fig, axes = plt.subplots(1, 2, figsize=(9, 4.2))
for ax, c, title in [(axes[0], "noop", "No-op start"), (axes[1], "sticky", "Sticky actions")]:
    for g in [g for g in GROUPS if g in set(tu.group)]:
        u = tu[tu.group == g]
        ax.scatter(u[f"tape_{c}_ret"], u[f"agent_{c}"], s=24, color=GROUP_STYLE[g][0], label=g, alpha=0.85, edgecolor="white", lw=0.5)
    hi = max(1.1, np.nanmax(tu[[f"tape_{c}_ret", f"agent_{c}"]].values) * 1.05)
    ax.plot([0, hi], [0, hi], color="#555", lw=0.8, ls="--"); ax.set_xlim(0, hi); ax.set_ylim(0, hi)
    ax.set_xlabel("blind-tape retention"); ax.set_title(title, fontsize=10)
axes[0].set_ylabel("agent retention"); axes[1].legend(frameon=False, fontsize=8, loc="lower right")
fig.suptitle("Agent vs. its own blind button tapes (above the line: the agent uses the screen)", fontsize=10)
fig.tight_layout(); save_fig(fig, "closed_loop_index"); plt.show()

## 5. What the encoders encode: a level-identity probe

A shared set of frames is collected on every level with a right-biased random policy, so all models are probed on identical inputs. Two measures, both with the HUD hidden:

- **Level-explained variance (η²)**: the share of each feature's variance across frames that is explained by which level the frame came from, averaged over active features. 0 = features look the same on every level; 1 = features only encode the level. This is the primary measure because it does not saturate.
- **Linear probe accuracy**: a linear classifier is trained on each encoder's 512-d features to predict which level a frame comes from; cross-validation folds are split by episode, so near-identical neighbouring frames never sit on both sides. Levels differ visibly (palettes, scenery), so even raw pixels identify the level almost perfectly; accuracy is reported for completeness.

The HUD (which prints `WORLD x-y`) is hidden, otherwise the level can simply be read off the screen. Representations that keep more level-specific information than the task needs are associated with overfitting (Garcin et al., 2023). Controls: an untrained encoder and raw downsampled pixels; chance is 1 / number of levels.

In [ ]:
PROBE_FILE = DIRS["raw"] / "probe_frames.npz"
RIGHT_ACTIONS = [1, 2, 3, 4]                       # right, right+A, right+B, right+A+B

def collect_probe(args):
    level, seed = args
    torch.set_num_threads(1)
    env, rng = make_mario(level), np.random.default_rng(seed)
    S, X, E = [], [], []
    for epi in range(PROBE_EPISODES):
        stack = np.zeros((120, 128, 4), np.uint8); stack[..., -1] = prep(mario_reset(env))
        kept, max_x, last_gain, a = 0, 0, 0, 1
        for t in range(MAX_STEPS):
            if t % 4 == 0:                         # hold each choice for 4 decisions; 80% of choices move right
                a = int(rng.choice(RIGHT_ACTIONS)) if rng.random() < 0.8 else int(rng.integers(N_ACTIONS))
            frame, info, done = env_step(env, a)
            stack[..., :-1] = stack[..., 1:]; stack[..., -1] = prep(frame)
            x = int(info.get("x_pos", 0))
            if x > max_x:
                max_x, last_gain = x, t
            if t >= 3 and t % PROBE_EVERY == 0:
                S.append(stack.copy()); X.append(x); E.append(epi); kept += 1
            if done or kept >= PROBE_MAX_PER_EPISODE or t - last_gain >= 60:
                break
    env.close()
    return level, np.array(S), np.array(X), np.array(E)

if PROBE_FILE.exists() and not REEVALUATE:
    z = np.load(PROBE_FILE, allow_pickle=False)
    P_STACKS, P_LEVEL, P_X, P_EP = z["stacks"], z["level"].astype(str), z["x"], z["ep"]
    print(f"using saved probe frames ({len(P_STACKS):,})")
else:
    res = run_pool(collect_probe, [(l, i) for i, l in enumerate(ALL_LEVELS)], "probe frames")
    res.sort(key=lambda r: ALL_LEVELS.index(r[0]))
    P_STACKS = np.concatenate([r[1] for r in res]); P_X = np.concatenate([r[2] for r in res])
    P_EP = np.concatenate([r[3] for r in res]); P_LEVEL = np.concatenate([[r[0]] * len(r[1]) for r in res]).astype(str)
    np.savez_compressed(PROBE_FILE, stacks=P_STACKS, level=P_LEVEL, x=P_X, ep=P_EP)
    print(f"saved → {RESULTS.name}/raw/probe_frames.npz ({len(P_STACKS):,} frames)")
P_MASKED = mask_hud(P_STACKS)
print(pd.Series(P_LEVEL).value_counts().reindex(ALL_LEVELS).to_frame("frames").T.to_string())

In [ ]:
@torch.no_grad()
def encode(policy, stacks, bs=256):
    out = []
    for i in range(0, len(stacks), bs):
        t, _ = policy.obs_to_tensor(stacks[i:i + bs])
        out.append(policy.extract_features(t).float().cpu())
    return torch.cat(out)

def probe_accuracy(Z, labels, groups, n_folds=5, wd=1e-3):
    """Multinomial logistic regression, folds split by probe episode. Returns mean held-out accuracy."""
    Z = torch.as_tensor(np.asarray(Z), dtype=torch.float32)
    classes = sorted(set(labels)); y = torch.tensor([classes.index(l) for l in labels])
    accs = []
    for f in range(n_folds):
        te = torch.as_tensor(groups % n_folds == f); tr = ~te
        if te.sum() == 0 or len(set(y[tr].tolist())) < len(classes):
            continue
        mu, sd = Z[tr].mean(0), Z[tr].std(0) + 1e-6
        Xtr, Xte = (Z[tr] - mu) / sd, (Z[te] - mu) / sd
        W = torch.zeros(Z.shape[1], len(classes), requires_grad=True); b = torch.zeros(len(classes), requires_grad=True)
        opt = torch.optim.LBFGS([W, b], lr=1, max_iter=300, line_search_fn="strong_wolfe")
        def closure():
            opt.zero_grad()
            loss = F.cross_entropy(Xtr @ W + b, y[tr]) + wd * W.pow(2).sum()
            loss.backward()
            return loss
        opt.step(closure)
        accs.append(((Xte @ W + b).argmax(1) == y[te]).float().mean().item())
    return float(np.mean(accs)) if accs else np.nan

def level_eta2(Z, labels):
    """Mean over active features of between-level variance ÷ total variance (one-way ANOVA η²)."""
    Z = np.asarray(Z, dtype=np.float64); labels = np.asarray(labels)
    tot = Z.var(0); active = tot > 1e-8 * max(tot.max(), 1e-12)
    if not active.any():
        return np.nan
    Z = Z[:, active]; mu = Z.mean(0)
    between = sum((labels == l).sum() * (Z[labels == l].mean(0) - mu) ** 2 for l in set(labels)) / len(Z)
    return float((between / Z.var(0)).mean())

torch.set_num_threads(max(1, os.cpu_count() or 1))
PROBE_FILE_RES = DIRS["raw"] / "probe_results.csv"
done = pd.read_csv(PROBE_FILE_RES) if PROBE_FILE_RES.exists() and not REEVALUATE else pd.DataFrame(columns=["model"])
rows = done.to_dict("records")
for r in MODELS.itertuples():
    if r.model in set(done.model):
        continue
    pol = PPO.load(r.path, device="cpu").policy; pol.set_training_mode(False)
    zm, zv = encode(pol, P_MASKED), encode(pol, P_STACKS)
    rows.append({"model": r.model, "family": r.family, "variant": r.variant, "seed": r.seed,
                 "eta2": level_eta2(zm, P_LEVEL), "acc_hud_hidden": probe_accuracy(zm, P_LEVEL, P_EP),
                 "acc_hud_visible": probe_accuracy(zv, P_LEVEL, P_EP)})
    pd.DataFrame(rows).to_csv(PROBE_FILE_RES, index=False)
    print(f"probe {r.model}: η² {rows[-1]['eta2']:.3f}, level-ID accuracy {rows[-1]['acc_hud_hidden']:.2f} (HUD hidden)", flush=True)

if "control_untrained" not in set(done.model):
    ref = PPO.load(MODELS.path.iloc[0], device="cpu")
    torch.manual_seed(0)
    fresh = type(ref.policy)(ref.observation_space, ref.action_space, lambda _: 0.0, **ref.policy_kwargs); fresh.set_training_mode(False)
    pix = lambda s: F.avg_pool2d(torch.as_tensor(s[..., -1]).float().unsqueeze(1), 8).flatten(1)   # 120×128 → 15×16
    for name, fm, fv in [("control_untrained", encode(fresh, P_MASKED), encode(fresh, P_STACKS)),
                         ("control_pixels", pix(P_MASKED), pix(P_STACKS))]:
        rows.append({"model": name, "family": "control", "variant": name.split("_")[1], "seed": 0, "eta2": level_eta2(fm, P_LEVEL),
                     "acc_hud_hidden": probe_accuracy(fm, P_LEVEL, P_EP), "acc_hud_visible": probe_accuracy(fv, P_LEVEL, P_EP)})
    pd.DataFrame(rows).to_csv(PROBE_FILE_RES, index=False)
PROBE = pd.DataFrame(rows)

n_cls = len(set(P_LEVEL))
agg = []
for (fam, var), d in PROBE.groupby(["family", "variant"], sort=False):
    label = {"untrained": "untrained encoder (control)", "pixels": "raw pixels 15×16 (control)"}.get(var, VARIANT_LABELS.get(var, var))
    agg.append({"Family": fam, "Encoder": label, "Models": len(d),
                "Level-explained variance η²": d.eta2.mean(), "± sd": d.eta2.std(ddof=1) if len(d) > 1 else np.nan,
                "Probe accuracy, HUD hidden": d.acc_hud_hidden.mean(), "Probe accuracy, HUD visible": d.acc_hud_visible.mean()})
save_table(pd.DataFrame(agg), "probe_level_identity",
           f"Level information in encoder features on shared frames ({n_cls} levels; probe chance {1 / n_cls:.2f}; folds split by episode)", ",.3f")

a = pd.DataFrame(agg)
fig, axes = plt.subplots(1, 2, figsize=(max(9, 1.3 * len(a) + 2), 3.6))
cols = ["#2a78d6" if f == "v1" else "#1baf7a" if f == "v2" else "#9a9a95" for f in a.Family]
ticks = [f"{f}: {e.replace(' (control)', '')}" for f, e in zip(a.Family, a.Encoder)]
axes[0].bar(range(len(a)), a["Level-explained variance η²"], color=cols, yerr=a["± sd"].fillna(0), capsize=2)
axes[0].set_ylabel("η² (share of feature variance)"); axes[0].set_title("Feature variance explained by level identity", fontsize=10)
axes[1].bar(range(len(a)), a["Probe accuracy, HUD hidden"], color=cols)
axes[1].axhline(1 / n_cls, color="#555", ls="--", lw=0.8)
axes[1].text(len(a) - 0.5, 1 / n_cls + 0.01, "chance", ha="right", fontsize=8, color="#555")
axes[1].set_ylim(0, 1.05); axes[1].set_ylabel("held-out accuracy"); axes[1].set_title("Linear level-identity probe", fontsize=10)
for ax in axes:
    ax.set_xticks(range(len(a)), ticks, fontsize=7, rotation=35, ha="right", rotation_mode="anchor")
fig.suptitle("How much do the encoders encode which level they are on? (HUD hidden; grey = controls)", fontsize=10)
fig.tight_layout(); save_fig(fig, "probe_level_identity"); plt.show()

v2u = UNITS[UNITS.family == "v2"]
if len(v2u):
    gap = (v2u[v2u.group == "v2 · training levels"].groupby("model").clean_score.mean()
           - v2u[v2u.group == "v2 · held-out levels"].groupby("model").clean_score.mean()).rename("gap")
    gp = PROBE.set_index("model").join(gap, how="inner")
    if len(gp) >= 3:
        rho, p = stats.spearmanr(gp.eta2, gp.gap)
        save_table(gp.reset_index()[["model", "variant", "seed", "eta2", "acc_hud_hidden", "gap"]].rename(
                   columns={"eta2": "Level η²", "acc_hud_hidden": "Probe accuracy", "gap": "Generalisation gap (train − held-out score)"}),
                   "probe_vs_gap", f"v2 models: level-explained variance vs. generalisation gap (Spearman ρ = {rho:.2f}, p = {p:.2f}, n = {len(gp)})", ",.3f")

## 6. Attention under narrow vs. broad training (identical frames)

All attention models see the same probe frames. **Row entropy**: how spread out each position's attention is (normalised, 1 = uniform over the 132 positions). **Effective positions**: exp(entropy), the number of positions a query effectively attends to. **HUD share**: fraction of all received attention that lands on the top row of cells, which covers the score/clock bar (uniform = 12 / 132 = 0.09). The architecture-matched comparison is v1 `spatial` (one level) vs. v2 `attn_aug` (16 levels, same dense block). Sparse attention is capped at 8 keys per head by construction, so its entropy is not comparable to the dense blocks.

In [ ]:
@torch.no_grad()
def attention_maps(policy, stacks, bs=128):
    blk = policy.features_extractor.block; blk.store_attn = True; out = []
    try:
        for i in range(0, len(stacks), bs):
            t, _ = policy.obs_to_tensor(stacks[i:i + bs]); policy.extract_features(t)
            out.append(blk.last_attn.float())
    finally:
        blk.store_attn = False
    return torch.cat(out)                                    # (N, positions, positions), rows sum to 1

FEAT_HW = None
rows = []
for name in ATTN_MODELS:
    r = MODELS.set_index("model").loc[name]
    pol = PPO.load(r.path, device="cpu").policy; pol.set_training_mode(False)
    FEAT_HW = pol.features_extractor.feat_chw[1:]
    A = attention_maps(pol, P_STACKS).clamp_min(1e-12)
    n = A.shape[-1]
    row_h = (-(A * A.log()).sum(-1)).mean(-1).numpy()                  # nats, mean over query positions
    recv = A.mean(1)                                                    # (N, positions) attention received
    hud = recv.view(-1, *FEAT_HW)[:, 0, :].sum(-1).numpy()
    for lvl in ALL_LEVELS:
        m = P_LEVEL == lvl
        rows.append({"model": name, "family": r.family, "variant": r.variant, "seed": r.seed, "block": r.block, "gamma": r.gamma,
                     "level": lvl, "split": "trained" if lvl in r.trained_levels else "unseen",
                     "row_entropy": row_h[m].mean() / np.log(n), "effective_positions": np.exp(row_h[m]).mean(), "hud_share": hud[m].mean()})
ATT = pd.DataFrame(rows)
if len(ATT):
    ATT.to_csv(DIRS["raw"] / "attention_shared_frames.csv", index=False)
    common = ATT[ATT.level.isin(V1_LEVELS)]
    rows = []
    for (fam, var, blk), d in common.groupby(["family", "variant", "block"], sort=False):
        rows.append({"Model": f"{fam} {VARIANT_LABELS.get(var, var)}", "Training levels": "1" if fam == "v1" else str(len(V2_TRAIN_LEVELS)),
                     "Seeds": d.seed.nunique(), "Row entropy": d.row_entropy.mean(), "Effective positions": d.effective_positions.mean(),
                     "HUD share": d.hud_share.mean(), "Row entropy, own level(s)": d[d.split == "trained"].row_entropy.mean(),
                     "Row entropy, unseen": d[d.split == "unseen"].row_entropy.mean(), "|γ|": d.gamma.mean()})
    save_table(pd.DataFrame(rows), "attention_shared_frames",
               f"Attention on identical frames from levels {', '.join(V1_LEVELS)} (entropy normalised; 1 = uniform; uniform HUD share = {FEAT_HW[1]}/{FEAT_HW[0] * FEAT_HW[1]})", ",.3f")

    fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.4))
    order = list(dict.fromkeys(common.family + " " + common.variant.map(lambda v: VARIANT_LABELS.get(v, v))))
    for ax, col, title, ref in [(axes[0], "row_entropy", "row entropy (1 = uniform)", None),
                                (axes[1], "hud_share", "share of attention on the HUD row", FEAT_HW[1] / (FEAT_HW[0] * FEAT_HW[1]))]:
        for i, key in enumerate(order):
            d = common[(common.family + " " + common.variant.map(lambda v: VARIANT_LABELS.get(v, v))) == key]
            per_model = d.groupby("model")[col].mean()
            c = "#2a78d6" if key.startswith("v1") else "#1baf7a"
            ax.bar(i, per_model.mean(), color=c, width=0.6)
            ax.scatter(np.full(len(per_model), i) + np.linspace(-0.12, 0.12, len(per_model)), per_model, s=12, color="#222", zorder=3)
        if ref is not None:
            ax.axhline(ref, color="#555", ls="--", lw=0.8); ax.text(len(order) - 0.5, ref, "uniform", ha="right", va="bottom", fontsize=8, color="#555")
        ax.set_xticks(range(len(order)), [k.replace(" ", "\n", 1) for k in order], fontsize=8); ax.set_title(title, fontsize=10)
    fig.suptitle(f"Attention on identical frames from {', '.join(V1_LEVELS)} (dots: individual models)", fontsize=10)
    fig.tight_layout(); save_fig(fig, "attention_shared_frames"); plt.show()
else:
    print("no attention models found")

## 7. Do the agents use what they attend to? Attention-guided occlusion

For each attention model and frame, the 8 feature-map cells that receive the most attention are occluded in the input (the matching ~11 × 11-px patches, filled with the frame's mean, in all 4 stacked frames). The policy change KL(π(·|s) ‖ π(·|s_occluded)) is compared with occluding 8 random cells (mean of 3 draws) and the 8 least-attended cells. If attention is a faithful explanation, the top cells should change the policy far more than random cells (Atrey et al., 2020; Jain & Wallace, 2019).

In [ ]:
def cell_boxes(hw, out_hw=(120, 128)):
    r = np.linspace(0, out_hw[0], hw[0] + 1).round().astype(int); c = np.linspace(0, out_hw[1], hw[1] + 1).round().astype(int)
    return [(r[i], r[i + 1], c[j], c[j + 1]) for i in range(hw[0]) for j in range(hw[1])]

def occlude(stacks, cells, boxes):
    out = stacks.copy(); fill = stacks.mean(axis=(1, 2)).astype(np.uint8)          # per frame and channel
    for i, cs in enumerate(cells):
        for c in cs:
            r0, r1, c0, c1 = boxes[c]; out[i, r0:r1, c0:c1, :] = fill[i]
    return out

def kl(p, q):
    p, q = np.clip(p, 1e-8, 1), np.clip(q, 1e-8, 1)
    return (p * np.log(p / q)).sum(-1)

rng = np.random.default_rng(0)
OCC_IDX = np.concatenate([rng.permutation(np.flatnonzero(P_LEVEL == l))[:OCCLUDE_FRAMES] for l in ALL_LEVELS])
rows, examples = [], None
for name in ATTN_MODELS:
    r = MODELS.set_index("model").loc[name]
    agent = Agent(r.path); pol = agent.policy
    hw = pol.features_extractor.feat_chw[1:]; boxes = cell_boxes(hw); n = hw[0] * hw[1]
    S = P_STACKS[OCC_IDX]
    recv = attention_maps(pol, S).mean(1).numpy()
    order = np.argsort(-recv, axis=1)
    top, bottom = order[:, :OCCLUDE_K], order[:, -OCCLUDE_K:]
    p0 = agent.probs(S)
    kl_top, kl_bottom = kl(p0, agent.probs(occlude(S, top, boxes))), kl(p0, agent.probs(occlude(S, bottom, boxes)))
    rand_sets = [np.array([rng.choice(n, OCCLUDE_K, replace=False) for _ in range(len(S))]) for _ in range(3)]
    kl_rand = np.mean([kl(p0, agent.probs(occlude(S, rs, boxes))) for rs in rand_sets], axis=0)
    flip_top = (agent.probs(occlude(S, top, boxes)).argmax(1) != p0.argmax(1))
    for i, idx in enumerate(OCC_IDX):
        rows.append({"model": name, "family": r.family, "variant": r.variant, "seed": r.seed, "level": P_LEVEL[idx],
                     "split": "trained" if P_LEVEL[idx] in r.trained_levels else "unseen",
                     "kl_top": kl_top[i], "kl_random": kl_rand[i], "kl_bottom": kl_bottom[i], "argmax_change_top": int(flip_top[i])})
    if examples is None:
        examples = (name, S[:: max(1, len(S) // 4)][:4], top[:: max(1, len(S) // 4)][:4], rand_sets[0][:: max(1, len(S) // 4)][:4], boxes)
OCC = pd.DataFrame(rows)
if len(OCC):
    OCC.to_csv(DIRS["raw"] / "occlusion_frames.csv", index=False)
    rows = []
    for (fam, var), d in OCC.groupby(["family", "variant"], sort=False):
        per = d.groupby("model")[["kl_top", "kl_random", "kl_bottom"]].mean()
        rows.append({"Model": f"{fam} {VARIANT_LABELS.get(var, var)}", "Models": len(per), "Frames": len(d),
                     "KL top-8 attended": d.kl_top.mean(), "KL random 8": d.kl_random.mean(), "KL least-attended 8": d.kl_bottom.mean(),
                     "Top ÷ random": d.kl_top.mean() / max(d.kl_random.mean(), 1e-12),
                     "Frames top > random": f"{(d.kl_top > d.kl_random).mean():.0%}",
                     "p (Wilcoxon, frames)": wilcoxon_p(d.kl_top, d.kl_random)})
    t = pd.DataFrame(rows); t["p (Holm)"] = holm(t["p (Wilcoxon, frames)"])
    save_table(t, "occlusion_faithfulness", "Policy change (KL) when occluding the 8 most-attended, 8 random or 8 least-attended cells", ",.4f")

    fig, ax = plt.subplots(figsize=(max(4.5, 1.6 * len(t) + 1.5), 3.6)); x = np.arange(len(t)); w = 0.26
    for k, (col, c, lab) in enumerate([("KL top-8 attended", "#eb6834", "most attended"), ("KL random 8", "#9a9a95", "random"),
                                       ("KL least-attended 8", "#2a78d6", "least attended")]):
        ax.bar(x + (k - 1) * w, t[col], w * 0.92, color=c, label=lab)
    ax.set_xticks(x, [m.replace(" ", "\n", 1) for m in t.Model], fontsize=8); ax.set_ylabel("KL(π ‖ π occluded)")
    ax.legend(frameon=False, fontsize=8); ax.set_title(f"Policy change when {OCCLUDE_K} cells are occluded", fontsize=10)
    fig.tight_layout(); save_fig(fig, "occlusion_faithfulness"); plt.show()

    name, S_ex, top_ex, rnd_ex, boxes = examples
    fig, axes = plt.subplots(1, len(S_ex), figsize=(3.2 * len(S_ex), 3.3), squeeze=False)
    for a, s_, tp, rd in zip(axes[0], S_ex, top_ex, rnd_ex):
        a.imshow(s_[..., -1], cmap="gray", vmin=0, vmax=255); a.axis("off")
        for cells, col, ls in [(tp, "#eb6834", "-"), (rd, "white", "--")]:
            for c in cells:
                r0, r1, c0, c1 = boxes[c]; a.add_patch(Rectangle((c0, r0), c1 - c0, r1 - r0, fill=False, ec=col, lw=1.2, ls=ls))
    fig.suptitle(f"{name}: {OCCLUDE_K} most-attended cells (orange) and {OCCLUDE_K} random cells (dashed)", fontsize=10)
    fig.tight_layout(); save_fig(fig, "occlusion_examples"); plt.show()

## 8. Test-time interventions on unseen levels (no training)

Can anything done at test time, without retraining, help on levels the agent never saw? Each model plays only its unseen levels with:

| Intervention | What it does | Why it might help |
|---|---|---|
| `hud_mask` | hide the score/coin/clock bar | Song et al. (2020) found a Sonic agent relied on the scoreboard/timer; hiding it raised test reward by about 10% |
| `tta` | average the policy over the frame and 4 copies shifted by ±4 px | smooths out decisions that hinge on exact pixel positions |
| `greedy` | always take the most likely action instead of sampling | removes random mistakes, but can loop |
| `attn_off` | set the attention gate γ to 0 (attention models only) | tests whether attention helps or hurts on unseen levels |

Each is compared with the same model's clean play on the same level (section 4). **Change** = intervention score ÷ clean score − 1.

In [ ]:
jobs = []
for r in MODELS.itertuples():
    levels = V1_LEVELS if r.family == "v1" else V2_TEST_LEVELS
    conds = [c for c in INTERVENTIONS if c != "attn_off" or r.has_attention]
    for lvl in levels:
        if lvl in r.trained_levels or not conds:
            continue
        jobs.append({"id": f"{r.model}__{lvl}", "subdir": "interventions", "model": r.model, "family": r.family,
                     "variant": r.variant, "seed": r.seed, "level": lvl,
                     "split": "unseen level" if r.family == "v1" else "held-out levels", "path": r.path,
                     "conditions": conds, "episodes": V1_EPISODES if r.family == "v1" else V2_EPISODES})
print(f"{len(jobs)} (model, unseen level) jobs, {sum(len(j['conditions']) * j['episodes'] for j in jobs):,} episodes")
IV = pd.concat(run_pool(eval_job, jobs, "interventions"), ignore_index=True) if jobs else pd.DataFrame()
if len(IV):
    IV["group"] = IV.family + " · " + IV.split
    IV.to_csv(DIRS["raw"] / "intervention_episodes.csv", index=False)
    iu = IV.pivot_table(index=["model", "variant", "level", "group"], columns="condition", values="max_x_pos", aggfunc="mean").reset_index()
    iu = iu.merge(UNITS[["model", "level", "clean"]], on=["model", "level"])
    rows = []
    for g in [g for g in GROUPS if g in set(iu.group)]:
        for c in [c for c in INTERVENTIONS if c in iu]:
            u = iu[(iu.group == g) & iu[c].notna()]
            if not len(u):
                continue
            ch = u[c] / u["clean"] - 1
            rows.append({"Group": g, "Intervention": COND_LABELS[c], "n": len(u),
                         "Clean score": (u["clean"] / u.level.map(RANDOM_MEAN)).mean(),
                         "Score with intervention": (u[c] / u.level.map(RANDOM_MEAN)).mean(),
                         "Change": mean_ci(100 * ch, "{:+.1f}%"), "Units improved": f"{(ch > 0).mean():.0%}",
                         "p (Wilcoxon)": wilcoxon_p(u[c], u["clean"])})
    INT_T = pd.DataFrame(rows); INT_T["p (Holm)"] = holm(INT_T["p (Wilcoxon)"])
    save_table(INT_T, "test_time_interventions", "Test-time interventions on unseen levels vs. clean play (change: mean, 95% CI over model × level)", ",.3f")
    rows = []
    for (g, v), u in iu.groupby(["group", "variant"]):
        rows.append({"Group": g, "Variant": VARIANT_LABELS.get(v, v), "Clean score": (u["clean"] / u.level.map(RANDOM_MEAN)).mean(),
                     **{COND_LABELS[c]: (u[c] / u.level.map(RANDOM_MEAN)).mean() for c in INTERVENTIONS if c in u}})
    save_table(pd.DataFrame(rows), "test_time_interventions_by_variant", "Unseen-level score (× random) per variant and intervention")

    fig, ax = plt.subplots(figsize=(1.9 * len(INTERVENTIONS) + 1.5, 3.6))
    gs = [g for g in GROUPS if g in set(iu.group)]; w = 0.8 / len(gs)
    for i, g in enumerate(gs):
        xs, ms, cis = [], [], []
        for j, c in enumerate(INTERVENTIONS):
            u = iu[(iu.group == g)] if c in iu else iu.iloc[:0]
            u = u[u[c].notna()] if c in iu else u
            if not len(u):
                continue
            ch = 100 * (u[c] / u["clean"] - 1)
            xs.append(j + (i - (len(gs) - 1) / 2) * w); ms.append(ch.mean()); cis.append(boot_ci(ch))
        if xs:
            col, hatch = GROUP_STYLE[g]; cis = np.array(cis)
            ax.bar(xs, ms, w * 0.92, color=col, hatch=hatch, edgecolor="white", linewidth=0, label=g,
                   yerr=np.abs(np.nan_to_num(cis.T - np.array(ms))), capsize=2, error_kw={"elinewidth": 0.8})
    ax.axhline(0, color="#555", lw=0.8)
    ax.set_xticks(range(len(INTERVENTIONS)), [COND_LABELS[c] for c in INTERVENTIONS])
    ax.set_ylabel("change in unseen-level max x (%)"); ax.legend(frameon=False, fontsize=8)
    ax.set_title("Test-time interventions on unseen levels (95% CI)", fontsize=10)
    fig.tight_layout(); save_fig(fig, "test_time_interventions"); plt.show()

## 9. Which training-level measurement predicts unseen-level performance?

If a quantity measured **only on the levels a model was trained on** predicts how well it plays **levels it never saw**, it can be used to choose models and detect memorisation without any test levels (the RL analogue of Jiang et al., 2020, *Fantastic Generalization Measures*). For each model:

| Measure | Computed on training levels only |
|---|---|
| Training-level score | clean max x ÷ random |
| No-op / sticky / frozen-HUD / shift retention | section 4 |
| **Closed-loop index** | agent retention − blind-tape retention under sticky actions or no-op starts (section 4); > 0 means the agent reacts to the screen |
| Level information η² | section 5 (shared frames, all levels) |
| Dormant neurons | share of ReLU units whose normalised mean activation ≤ τ (Sokar et al., 2023) |
| Policy entropy | how undecided the policy is on its own levels (1 = uniform over 7 actions) |
| Shift sensitivity | KL between the policy on a frame and on the same frame shifted 6 px |
| Attention entropy | attention models only (section 6) |

Target: clean score on unseen levels. Statistics use the 27 v1 models (9 per training level); v2 models are shown but too few to test. **Within-level ρ** removes the training-level mean from both quantities first, so a correlation cannot come only from "1-2 models are better than 1-1 models".

In [ ]:
@torch.no_grad()
def dormant_fraction(policy, stacks, tau, bs=256):
    mods = [m for m in policy.features_extractor.modules() if isinstance(m, torch.nn.ReLU)]
    sums, n = [None] * len(mods), 0
    def hook(i):
        def f(_m, _inp, out):
            a = (out.abs().mean(dim=(2, 3)) if out.dim() == 4 else out.abs()).sum(0)
            sums[i] = a if sums[i] is None else sums[i] + a
        return f
    hs = [m.register_forward_hook(hook(i)) for i, m in enumerate(mods)]
    try:
        for i in range(0, len(stacks), bs):
            t, _ = policy.obs_to_tensor(stacks[i:i + bs]); policy.extract_features(t); n += len(t)
    finally:
        for h in hs:
            h.remove()
    scores = [s / n / max(float((s / n).mean()), 1e-12) for s in sums if s is not None]
    return float(np.mean(np.concatenate([(s <= tau).numpy() for s in scores]))) if scores else np.nan

rng = np.random.default_rng(1)
rows = []
for r in MODELS.itertuples():
    idx = np.flatnonzero(np.isin(P_LEVEL, r.trained_levels))
    idx = rng.permutation(idx)[:600]
    if not len(idx):
        continue
    agent = Agent(r.path); S = P_STACKS[idx]
    p0 = agent.probs(S)
    ps = agent.probs(np.stack([shift2d(s, 0, SHIFT_PX) for s in S]))
    rows.append({"model": r.model, "dormant": dormant_fraction(agent.policy, S, DORMANT_TAU),
                 "policy_entropy": float((-(np.clip(p0, 1e-12, 1) * np.log(np.clip(p0, 1e-12, 1))).sum(1)).mean() / np.log(N_ACTIONS)),
                 "shift_kl": float(kl(p0, ps).mean())})
FRAME_MEAS = pd.DataFrame(rows)

trained_g = UNITS[UNITS.group.isin(["v1 · trained level", "v2 · training levels"])].groupby("model")
unseen_g = UNITS[UNITS.group.isin(["v1 · unseen level", "v2 · held-out levels"])].groupby("model")
M = MODELS[["model", "family", "variant", "seed"]].copy()
M["trained_on"] = MODELS.trained_levels.map(lambda l: l[0] if len(l) == 1 else f"{len(l)} levels")
M = M.set_index("model")
M["unseen_score"] = unseen_g.clean_score.mean(); M["train_score"] = trained_g.clean_score.mean()
for c in PERTURB:
    M[f"ret_{c}"] = trained_g[f"ret_{c}"].mean()
if "CLI" in globals() and len(CLI):
    M["cli_sticky"] = CLI.groupby("model").cli_sticky.mean()
    M["cli_noop"] = CLI.groupby("model").cli_noop.mean()
M["eta2"] = PROBE.set_index("model").eta2
if len(ATT):
    M["attn_entropy"] = ATT[ATT.split == "trained"].groupby("model").row_entropy.mean()
M = M.join(FRAME_MEAS.set_index("model"))
M = M.reset_index()
save_table(M, "predictors_per_model", "Training-level measurements and unseen-level score for every model", ",.3f")

MEASURES = {"train_score": "Training-level score", "ret_noop": "No-op retention", "ret_sticky": "Sticky-action retention",
            "ret_hud_freeze": "Frozen-HUD retention", "ret_shift": "Shift retention", "cli_sticky": "Closed-loop index (sticky)",
            "cli_noop": "Closed-loop index (no-op)",
            "eta2": "Level information η²", "dormant": f"Dormant neurons (τ = {DORMANT_TAU})", "policy_entropy": "Policy entropy",
            "shift_kl": "Shift sensitivity (KL)", "attn_entropy": "Attention entropy"}
MEASURES = {k: v for k, v in MEASURES.items() if k in M and M[k].notna().sum() >= 3}

def spearman_ci(x, y, n_boot=5000, seed=0):
    x, y = np.asarray(x, float), np.asarray(y, float)
    rng = np.random.default_rng(seed); vals = []
    for _ in range(n_boot):
        i = rng.integers(0, len(x), len(x))
        if np.ptp(x[i]) > 0 and np.ptp(y[i]) > 0:
            vals.append(stats.spearmanr(x[i], y[i])[0])
    return np.percentile(vals, [2.5, 97.5]) if vals else (np.nan, np.nan)

v1m = M[M.family == "v1"]
rows = []
for k, label in MEASURES.items():
    d = v1m[[k, "unseen_score", "trained_on"]].dropna()
    if len(d) < 5 or d[k].nunique() < 3:
        continue
    rho, p = stats.spearmanr(d[k], d.unseen_score); lo, hi = spearman_ci(d[k], d.unseen_score)
    dx = d[k] - d.groupby("trained_on")[k].transform("mean"); dy = d.unseen_score - d.groupby("trained_on").unseen_score.transform("mean")
    rho_w, p_w = stats.spearmanr(dx, dy)
    rows.append({"Measure (training levels)": label, "n models": len(d), "Spearman ρ": rho, "95% CI": f"{lo:.2f} to {hi:.2f}",
                 "p": p, "Within-level ρ": rho_w, "Within-level p": p_w})
PRED = pd.DataFrame(rows)
if len(PRED):
    PRED["p (Holm)"] = holm(PRED["p"]); PRED["Within-level p (Holm)"] = holm(PRED["Within-level p"])
    PRED = PRED.sort_values("Spearman ρ", key=lambda s: -s.abs())
    save_table(PRED, "predictors_of_transfer",
               "Do training-level measurements predict unseen-level score? (v1 models; Holm over measures)", ",.3f")

    keys = [k for k in MEASURES if k in set(PRED["Measure (training levels)"].map({v: k for k, v in MEASURES.items()}))]
    ncol = 4; nrow = int(np.ceil(len(keys) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.3 * ncol, 2.9 * nrow), squeeze=False)
    lvl_col = dict(zip(V1_LEVELS, ["#2a78d6", "#eb6834", "#1baf7a", "#6250d6"]))
    for ax, k in zip(axes.flat, keys):
        for lvl, d in v1m.groupby("trained_on"):
            ax.scatter(d[k], d.unseen_score, s=18, color=lvl_col.get(lvl, "#555"), label=f"v1 trained on {lvl}", alpha=0.85)
        d2 = M[M.family == "v2"]
        if d2[k].notna().any():
            ax.scatter(d2[k], d2.unseen_score, s=26, marker="D", facecolor="none", edgecolor="#222", label="v2 (16 levels)")
        row = PRED[PRED["Measure (training levels)"] == MEASURES[k]].iloc[0]
        ax.set_title(f"{MEASURES[k]}\nρ = {row['Spearman ρ']:.2f}, within-level ρ = {row['Within-level ρ']:.2f}", fontsize=8)
        ax.axhline(1, color="#bbb", lw=0.7, ls="--")
    for ax in axes.flat[len(keys):]:
        ax.axis("off")
    for ax in axes[:, 0]:
        ax.set_ylabel("unseen-level score (× random)", fontsize=8)
    h, l = axes.flat[0].get_legend_handles_labels()
    fig.legend(h, l, loc="lower center", ncol=len(l), frameon=False, fontsize=8, bbox_to_anchor=(0.5, -0.02))
    fig.suptitle("Training-level measurements vs. unseen-level score (dashed: random play)", fontsize=10)
    fig.tight_layout(rect=(0, 0.04, 1, 1)); save_fig(fig, "predictors_of_transfer"); plt.show()
else:
    print("fewer than 5 v1 models with unseen-level results: correlations skipped")

## 10. Summary

In [ ]:
def val(df, row_filter, col):
    d = df.loc[row_filter, col]
    return float(d.mean()) if len(d) else np.nan

key = []
for g in groups:
    u = UNITS[UNITS.group == g]
    key.append(f"- **{g}** (n = {len(u)}): clean score {u.clean_score.mean():.2f}× random; retention " +
               ", ".join(f"{COND_LABELS[c].lower()} {u[f'ret_{c}'].mean():.2f}" for c in PERTURB) + ".")
if "rr" in globals() and len(rr):
    for g in rr.group.unique():
        u = rr[rr.group == g]
        key.append(f"- **{g}, no-op starts**: agent keeps {u.agent_retention.mean():.2f} of its clean progress, its blind replay "
                   f"{u.replay_retention.mean():.2f}; agent ahead in {(u.agent_retention > u.replay_retention).mean():.0%} of {len(u)} cases.")
if "CLI" in globals() and len(CLI):
    for g in [g for g in GROUPS if g in set(CLI.group)]:
        u = CLI[CLI.group == g]
        key.append(f"- **{g}, closed-loop index**: sticky {u.cli_sticky.mean():+.2f} (agent {u.agent_sticky.mean():.2f} vs tape "
                   f"{u.tape_sticky_ret.mean():.2f}), no-op {u.cli_noop.mean():+.2f}.")
if len(PROBE):
    for (fam, var), d in PROBE.groupby(["family", "variant"], sort=False):
        key.append(f"- Level information, {fam} {var}: η² {d.eta2.mean():.3f}, probe accuracy {d.acc_hud_hidden.mean():.2f} "
                   f"(HUD hidden; chance {1 / n_cls:.2f}).")
if len(ATT):
    for (fam, var), d in ATT[ATT.level.isin(V1_LEVELS)].groupby(["family", "variant"], sort=False):
        key.append(f"- Attention, {fam} {var}: row entropy {d.row_entropy.mean():.2f}, HUD share {d.hud_share.mean():.2f}.")
if len(OCC):
    for (fam, var), d in OCC.groupby(["family", "variant"], sort=False):
        key.append(f"- Occlusion, {fam} {var}: KL top-8 {d.kl_top.mean():.4f} vs random {d.kl_random.mean():.4f} "
                   f"(top > random on {(d.kl_top > d.kl_random).mean():.0%} of frames).")

if "INT_T" in globals() and len(INT_T):
    for _, r in INT_T.iterrows():
        key.append(f"- Test-time {r['Intervention'].lower()}, {r['Group']}: unseen-level change {r['Change']}, "
                   f"improved in {r['Units improved']} of {r['n']} (Holm p = {r['p (Holm)']:.3f}).")
if "PRED" in globals() and len(PRED):
    for _, r in PRED.head(4).iterrows():
        key.append(f"- Predictor of transfer, {r['Measure (training levels)']}: ρ = {r['Spearman ρ']:.2f} ({r['95% CI']}), "
                   f"within-level ρ = {r['Within-level ρ']:.2f} (Holm p = {r['p (Holm)']:.3f}).")

order = ["models", "robustness_summary", "robustness_statistics", "robustness_by_variant", "episode_endings", "replay_comparison",
         "closed_loop_index",
         "probe_level_identity", "probe_vs_gap", "attention_shared_frames", "occlusion_faithfulness",
         "test_time_interventions", "test_time_interventions_by_variant", "predictors_of_transfer", "predictors_per_model"]
lines = [f"# {TITLE}", "", f"Generated {time.strftime('%Y-%m-%d %H:%M')} · {len(MODELS)} models · levels {ALL_LEVELS}", "",
         "## Key numbers", "", *key, ""]
for name in order:
    p = DIRS["tables"] / f"{name}.md"
    if p.exists():
        lines += [p.read_text(), ""]
lines += ["## Figures", ""] + [f"![{p.stem}](figures/{p.name})" for p in sorted(DIRS["figures"].glob("*.png"))]
(RESULTS / "summary.md").write_text("\n".join(lines) + "\n")
print("\n".join(key))
print(f"\nsaved → {RESULTS.name}/summary.md")